# Packed SFT 的 Attention 问题

第 5 章的默认 SDPA 适合一条训练序列只放一条样本的情况。本节引入第 6 章要解决的问题：SFT 样本长度不一，为了减少 padding，我们把多个样本装进同一个定长容器；此时普通 causal mask 会让后面的样本读到前面的样本。

本节只做问题定义和实验分层，DataLoader 细节放在 06.03，第一次 profiling 放在 06.04，backend 配置放在 06.05，最终 profiling 放在 06.06。


## 1. 为什么要 packing

假设 `seq_len=1024`，四条真实样本长度为 `[180, 260, 410, 120]`。non-greedy 路线为每条样本分配完整 1024 slots；greedy 路线则尝试把它们装入更少的 containers。


In [ ]:
SEQ_LEN = 1024
sample_lengths = [180, 260, 410, 120]

non_greedy_slots = len(sample_lengths) * SEQ_LEN
greedy_slots = ((sum(sample_lengths) + SEQ_LEN - 1) // SEQ_LEN) * SEQ_LEN
useful_tokens = sum(sample_lengths)

print(f'useful tokens: {useful_tokens}')
print(f'non-greedy slots: {non_greedy_slots}, utilization={useful_tokens / non_greedy_slots:.1%}')
print(f'greedy slots: {greedy_slots}, utilization={useful_tokens / greedy_slots:.1%}')


packing 节省了 padding slots，但一个 container 现在包含多个原始文档。接下来必须同时满足：

- 同一文档内仍然保持 causal 可见性；
- 不同文档之间完全不可见；
- 每条样本内部的 multi-turn messages 不能被错误拆开。


## 2. 普通 causal mask 的语义漏洞

用 document ids `[0, 0, 0, 1, 1, 1]` 表示两个 packed 文档。普通 causal 规则只判断 `key <= query`，因此 document 1 的 query 可以读取 document 0 的 key。


In [ ]:
document_ids = [0, 0, 0, 1, 1, 1]
plain_causal = [
    [key <= query for key in range(len(document_ids))]
    for query in range(len(document_ids))
]
block_causal = [
    [
        key <= query and document_ids[key] == document_ids[query]
        for key in range(len(document_ids))
    ]
    for query in range(len(document_ids))
]

print('q4 -> k1 with causal:', plain_causal[4][1])
print('q4 -> k1 with block-causal:', block_causal[4][1])
assert plain_causal[4][1] is True
assert block_causal[4][1] is False


这不是单纯的性能问题。labels 设为 `-100` 只决定哪些位置贡献 loss，不能阻止 hidden states 作为其他文档的 K/V。因此 A 路线只能作为错误演示，不能作为优化 benchmark。


## 3. 四条路线与三类比较

| ID | Attention | Packing | 语义 | 用途 |
|---|---|---|---|---|
| A | SDPA causal | greedy | ❌ 跨文档可见 | correctness 反例 |
| B | SDPA causal | non-greedy | ✅ | 生产端到端 baseline |
| C | SDPA block-causal | greedy | ✅ | 相同 packing 下的 dense baseline |
| D | NPU Varlen block-causal | greedy | ✅ | TND/sparse 优化路线 |

比较关系：

- **B → D**：端到端总收益，包含 packing 和 backend 变化；
- **B → C**：引入 greedy packing 与 document mask 的系统变化；
- **C → D**：保持相同 packing 与语义，只观察 Varlen backend 的增量收益。


## 4. Qwen multi-turn：为什么不能用 EOS 分样本

Qwen3 的 `eos_id=151645` 对应 `<|im_end|>`，chat template 会在 user、assistant 等每条消息后写入它。Wordle 一条样本又包含多轮消息。因此，扫描所有 `eos_id` 可能得到 message boundaries，而不是 sample boundaries。

目标契约必须是：

```text
同一 Wordle sample 内：多轮消息彼此可见
不同 Wordle samples 之间：完全不可见
```

本教程使用的 SFT 训练路径不再扫描 EOS。Packing 时，每放入一条完整 Wordle sample，DataLoader 都让它的位置编号从 0 重新开始；trainer 看到编号再次变成 0，就知道下一条样本开始了。这样同一样本内部出现多少个 `<|im_end|>` 都不会把它拆开。


## 5. 本节结论

问题已经分成两层：

1. **语义层**：greedy packing 需要 block-causal，而不是整段 causal；
2. **计算层**：block-causal 逻辑上减少可见 pairs，但 dense SDPA 未必跳过 masked tiles；Varlen 用 `cu_seqlens` 和 TND backend 表达可计算区域。

下一节用实际的 positions 输出说明 DataLoader 如何保留这些样本起点，以及 trainer 为什么仍需要据此生成 block-causal mask 或 VarLen metadata。


## 练习

1. （判断题）把 prompt token 的 label 设为 IGNORE_INDEX，会同时阻止其他样本的 token 读取它的 K/V。

2. （判断题）多轮聊天中每个 EOS 都可以可靠地作为 packed sample 边界。

3. （单选题）多个独立样本被装进同一个 container 后，普通 causal mask 的主要问题是什么？
    A. 后面的样本可以读取前面样本的 hidden states
    B. 前面的样本可以读取未来 token
    C. tokenizer 会自动删除 EOS
    D. loss 会自动变成 NaN

4. （单选题）以下哪条路线既表达正确的样本隔离语义，又能用累计长度跳过跨样本 Attention pair？
    A. Greedy packing + 普通 causal SDPA
    B. Non-greedy + 只修改 labels
    C. Greedy packing + dense block-causal SDPA
    D. Greedy packing + TND VarLen Attention

In [ ]:
!cat ./answer/06.02_answer.txt
